
# XGBoost Random Forest

*Experiment 1. Random Forest*

In [ ]:
import os
from enum import Enum
from typing import Dict, List, Optional, Tuple, Union
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import clear_output
from tqdm.auto import tqdm

import polars as pl
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
import dask.array as da
import numpy as np
import xgboost as xgb
from xgboost import dask as dxgb

from oldutils.datasets import (
    ARTIFACTS_FOLDER,
    ROOT_FOLDER,
    STATIC_FEATURES,
    HydroFiles,
    HydroStaticFeaturesFiles,
    MeteoSeriesFeaturesFiles,
)
from oldutils.types import TimeRange

%matplotlib inline
os.chdir(ROOT_FOLDER)


## Datasets

In [3]:
PATH_MERGED_DATASETS = ARTIFACTS_FOLDER / "merged_datasets"
FILENAME_TRAIN_IDS = "train_file_ids.csv"

COL_GAUGE_ID = "gauge_id"
TARGETs = ["q_mm_day"]

HORIZON_HISTORY = TimeRange.YEAR
HORIZON_FORECAST = TimeRange.WEEK

In [4]:
def load_train_subset() -> dd.DataFrame:
    train_df = pl.read_csv(PATH_MERGED_DATASETS / FILENAME_TRAIN_IDS).sample(10)
    file_ids = train_df["file_id"].to_list()
    paths = [PATH_MERGED_DATASETS / f"{file_id}.parquet" for file_id in file_ids]
    return dd.read_parquet(paths)


def gen_lag_name(column: str, lag: int) -> str:
    return f"{column}_{lag}"


def gen_add_lags(column: str, lags: range, store_new_columns: list | None = None):
    for lag in lags:
        col_name = gen_lag_name(column, lag)
        if store_new_columns is not None:
            store_new_columns.append(col_name)

    def add_lags(pdf: pd.DataFrame) -> pd.DataFrame:
        pdf = pdf.sort_values(by=["date"])
        for lag in lags:
            pdf[gen_lag_name(column, lag)] = pdf[column].shift(lag)
        return pdf

    return add_lags


def create_lags(
    df: dd.DataFrame,
    lags_columns: list[str],
    lags: range,
    store_new_columns: list | None = None,
) -> dd.DataFrame:
    for column in lags_columns:
        df = df.map_partitions(gen_add_lags(column, lags, store_new_columns))
    return df


### Load datasets

In [6]:
import pandas as pd
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

In [7]:
df = load_train_subset().set_index(COL_GAUGE_ID)

lags = range(1, HORIZON_HISTORY + 1)
lags_columns = ["prcp", "t_max", "t_min", "t_mean", "q_mm_day", "lvl_sm"]

df = create_lags(df, lags_columns, lags)
targets = []
df = create_lags(df, TARGETs, range(1, HORIZON_FORECAST + 1), targets)
targets.extend(TARGETs)

df = df.dropna()

X = df.drop(columns=set(lags_columns + targets))
y = df[targets]
X_train, y_train = X, y

In [ ]:
hsff = HydroStaticFeaturesFiles()
for i in hsff:
    print(i[1].collect())
    break

shape: (1, 150)
┌──────────┬────────────┬────────────┬────────────┬───┬────────────┬────────────┬───────────┬──────┐
│ gauge_id ┆ for_pc_sse ┆ crp_pc_sse ┆ inu_pc_ult ┆ … ┆ ero_kh_sav ┆ hft_ix_s93 ┆ hft_ix_s0 ┆ acc  │
│ ---      ┆ ---        ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---        ┆ 9         ┆ ---  │
│ i64      ┆ f64        ┆ f64        ┆ f64        ┆   ┆ f64        ┆ f64        ┆ ---       ┆ f64  │
│          ┆            ┆            ┆            ┆   ┆            ┆            ┆ f64       ┆      │
╞══════════╪════════════╪════════════╪════════════╪═══╪════════════╪════════════╪═══════════╪══════╡
│ 1001     ┆ 56.666633  ┆ 0.0        ┆ 5.804287   ┆ … ┆ null       ┆ null       ┆ null      ┆ null │
└──────────┴────────────┴────────────┴────────────┴───┴────────────┴────────────┴───────────┴──────┘



### Model training

In [ ]:
# cluster = LocalCluster(n_workers=4, threads_per_worker=2, memory_limit="4GB")
# client = Client(cluster)
client = Client(asynchronous=False)

/home/khuzin/.cache/pypoetry/virtualenvs/flood-forecasts-Udo2UZ-X-py3.11/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 57474 instead
  warnings.warn(


In [ ]:
params = {
    "tree_method": "hist",
    "learning_rate": 0.1,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
}
EPS = 1e-6


def mape_obj(preds, dmat):
    y = dmat.get_label()
    grad = np.sign(preds - y) / (np.abs(y) + EPS)
    hess = EPS / (np.abs(y) + EPS) ** 2
    return grad, hess


async def main():
    cX_train = await client.compute(X_train)
    cy_train = await client.compute(y_train)
    dtrain = await dxgb.DaskDMatrix(client, cX_train, cy_train)
    result = await dxgb.train(
        client,
        params,
        dtrain,
        num_boost_round=500,
        obj=mape_obj,
    )
    print(result["history"])


await main()

# def main():
#     dtrain = dxgb.DaskDMatrix(client, X_train, y_train)
#     result = dxgb.train(
#         client,
#         params,
#         dtrain,
#         num_boost_round=500,
#         obj=mape_obj,
#     )
#     print(result["history"])

# main()

/tmp/ipykernel_1162/664106004.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
/tmp/ipykernel_1162/664106004.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
/tmp/ipykernel_1162/664106004.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
/tmp/ipykernel_1162/664106004.py:21: PerformanceWarning: D

AttributeError: 'DataFrame' object has no attribute '__await__'

In [26]:
import gc

client.close()
# cluster.close()
gc.collect()

1729

In [27]:
from dask.distributed import Client
import dask.dataframe as dd
from xgboost.dask import DaskDMatrix, train
import pandas as pd
import numpy as np

# Start a Dask client (using a local cluster).
client = Client()

# Step 1: Create a random dataset.
# Here we create a pandas DataFrame with a single column "value" filled with random numbers.
n_rows = 10000  # Adjust the number of rows as needed.
pdf = pd.DataFrame({'value': np.random.randn(n_rows)})

# Convert the pandas DataFrame to a Dask DataFrame with at least 10 partitions.
df = dd.from_pandas(pdf, npartitions=10)
print(f"Number of partitions: {df.npartitions}")

# Step 2: Define and apply a function to add shifted columns.
def add_shifts(pdf):
    # Compute shifted values within each partition.
    pdf["shift1"] = pdf["value"].shift(1)  # shift by 1
    pdf["shift2"] = pdf["value"].shift(2)  # shift by 2
    return pdf

# Apply the shifting function to each partition.
df = df.map_partitions(add_shifts)

# Remove rows with NaN values (these occur in the first two rows of each partition after shifting).
df = df.dropna()

# Step 3: Prepare the features and the target.
# We'll predict the original "value" based on "shift1" and "shift2".
X = df[["shift1", "shift2"]]
y = df["value"]

# Create the DaskDMatrix, so the data is streamed as needed to avoid loading everything into RAM.
dtrain = DaskDMatrix(client, X, y)

# Define parameters for an XGBoost regression model.
params = {
    "objective": "reg:squarederror",
    "max_depth": 3,
    "eta": 0.1,
}

# Train the model using XGBoost's Dask integration. Adjust num_boost_round as necessary.
output = train(client, params=params, dtrain=dtrain, num_boost_round=10)

print("Training metrics and booster information:")
print(output)


/home/khuzin/.cache/pypoetry/virtualenvs/flood-forecasts-Udo2UZ-X-py3.11/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 57272 instead
  warnings.warn(


Number of partitions: 10


[15:53:03] Task [xgboost.dask-0]:tcp://127.0.0.1:56950 got rank 0
[15:53:03] Task [xgboost.dask-3]:tcp://127.0.0.1:55824 got rank 1
2025-05-03 15:53:03,673 - distributed.worker - ERROR - Compute Failed
Key:       fn-c151195d-5af7-45b3-bfd3-4f0459995543
State:     long-running
Task:  <Task 'fn-c151195d-5af7-45b3-bfd3-4f0459995543' fn(...)>
Exception: 'TypeError("Unknown type: <class \'distributed.client.Future\'>")'
Traceback: '  File "/home/khuzin/.cache/pypoetry/virtualenvs/flood-forecasts-Udo2UZ-X-py3.11/lib/python3.11/site-packages/xgboost/dask/__init__.py", line 541, in fn\n    return [func(*args, **kwargs)]\n            ^^^^^^^^^^^^^^^^^^^^^\n  File "/home/khuzin/.cache/pypoetry/virtualenvs/flood-forecasts-Udo2UZ-X-py3.11/lib/python3.11/site-packages/xgboost/dask/__init__.py", line 799, in do_train\n    Xy, evals = _get_dmatrices(\n                ^^^^^^^^^^^^^^^\n  File "/home/khuzin/.cache/pypoetry/virtualenvs/flood-forecasts-Udo2UZ-X-py3.11/lib/python3.11/site-packages/xgboost/

TypeError: Unknown type: <class 'distributed.client.Future'>